In [1]:
import os
import pickle

import pandas as pd
import torch
from datasets import Dataset, concatenate_datasets, load_dataset
from huggingface_hub import hf_hub_download
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)

from trl import SFTTrainer, SFTConfig
from llamaAccessToken import LLamaToken

<h3 style="text-align:center"> downloading dataset</h3>

In [2]:
# Load the dataset
dataset = load_dataset("socratesft/SocSci210", split="train[:100000]") 

# Load a mapping file
participant_mapping = hf_hub_download(
    repo_id="socratesft/SocSci210",
    filename="metadata/participant_mapping.json",
    repo_type="dataset"
)

condition_mapping = hf_hub_download(
    repo_id="socratesft/SocSci210",
    filename="metadata/condition_mapping.json",
    repo_type="dataset"
)

task_mapping = hf_hub_download(
    repo_id="socratesft/SocSci210",
    filename="metadata/task_mapping.json",
    repo_type="dataset"
)

print(condition_mapping)

Repo card metadata block was not found. Setting CardData to empty.


Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

/home/jambo/.cache/huggingface/hub/datasets--socratesft--SocSci210/snapshots/048481111a4425ed83dc0eacf15f8431f252b21a/metadata/condition_mapping.json


<h3 style="text-align:center"> checking to see if cuda is installed </h3>

In [3]:
if torch.cuda.is_available():
    print("Cuda")
else:
    print("CPU")

Cuda


<h3 style="text-align:center"> Preprocessing dataset </h3>

In [4]:
print(type(dataset))

<class 'datasets.arrow_dataset.Dataset'>


In [5]:
dataset = dataset.to_pandas()

In [6]:
dataset.head()

,sample_id,participant,demographic,stimuli,response,condition_num,task_num,prompt,reasoning,study_id
0,0,0,"{'age': 34, 'education': 'Post grad study/prof...","You read ""Jaime is 20 years old and was born a...",5,7,0,You are a survey respondent with the following...,"Upon encountering the information about Jaime,...",9nphm
1,1,0,"{'age': 34, 'education': 'Post grad study/prof...","You read ""Jaime is 20 years old and was born a...",3,7,1,You are a survey respondent with the following...,"Upon reading the scenario about Jaime, the res...",9nphm
2,2,1,"{'age': 67, 'education': 'Bachelor's degree', ...","You read ""Jaime is 20 years old and was born a...",6,6,0,You are a survey respondent with the following...,"Upon reading the description of Jaime, this re...",9nphm
3,3,1,"{'age': 67, 'education': 'Bachelor's degree', ...","You read ""Jaime is 20 years old and was born a...",5,6,1,You are a survey respondent with the following...,As an individual in their late 60s with a some...,9nphm
4,4,2,"{'age': 79, 'education': 'Post grad study/prof...","You read ""Jaime is 20 years old and was born a...",6,4,0,You are a survey respondent with the following...,The respondent is likely to reflect on their o...,9nphm


<p> Breaking down the demographic column into its own dataframe</p>

In [7]:
demographicDataframe = pd.concat([
    dataset[["study_id"]].reset_index(drop=True),
    pd.DataFrame(dataset["demographic"].tolist())
], axis=1)

In [8]:
print(demographicDataframe)

      study_id  age                            education  \
0        9nphm   34  Post grad study/professional degree   
1        9nphm   34  Post grad study/professional degree   
2        9nphm   67                    Bachelor's degree   
3        9nphm   67                    Bachelor's degree   
4        9nphm   79  Post grad study/professional degree   
...        ...  ...                                  ...   
99995    kryns   65   High school graduate or equivalent   
99996    kryns   65   High school graduate or equivalent   
99997    kryns   65   High school graduate or equivalent   
99998    kryns   65   High school graduate or equivalent   
99999    kryns   65   High school graduate or equivalent   

                      employment ethnicity  gender  household_size  \
0      Employed as paid employee       NaN  Female               4   
1      Employed as paid employee       NaN  Female               4   
2                        Retired       NaN    Male               2   

<p> to solve the na values problem i decide to fill in values from the previous row </p>

In [9]:
demographicDataframe = demographicDataframe.ffill()

In [10]:
print(demographicDataframe.isna().sum())

study_id                 0
age                      0
education                0
employment               0
ethnicity            16407
gender                   0
household_size           0
housing_ownership        0
housing_type             0
ideology                 0
income                   0
internet_access          0
location                 0
marital_status           0
metro_status             0
party_id                 0
phone_service            0
dtype: int64


noticing ethnicity is full of NA's so will apply backwards fill to it

In [11]:
demographicDataframe["ethnicity"] = demographicDataframe["ethnicity"].bfill()

In [12]:
print(demographicDataframe.isna().sum())

study_id             0
age                  0
education            0
employment           0
ethnicity            0
gender               0
household_size       0
housing_ownership    0
housing_type         0
ideology             0
income               0
internet_access      0
location             0
marital_status       0
metro_status         0
party_id             0
phone_service        0
dtype: int64


In [13]:
demographicDataframe.head()

,study_id,age,education,employment,ethnicity,gender,household_size,housing_ownership,housing_type,ideology,income,internet_access,location,marital_status,metro_status,party_id,phone_service
0,9nphm,34,Post grad study/professional degree,Employed as paid employee,White,Female,4,Owned or being bought by you or someone in you...,A building with 2 or more apartments,Moderate,75-99K,Internet Household,New Jersey,Married,Metro Area,Moderate Democrat,Cellphone only
1,9nphm,34,Post grad study/professional degree,Employed as paid employee,White,Female,4,Owned or being bought by you or someone in you...,A building with 2 or more apartments,Moderate,75-99K,Internet Household,New Jersey,Married,Metro Area,Moderate Democrat,Cellphone only
2,9nphm,67,Bachelor's degree,Retired,White,Male,2,Owned or being bought by you or someone in you...,A one-family house attached to one or more houses,Somewhat Conservative,150-175K+,Internet Household,Iowa,Married,Metro Area,Don't Lean/Independent/None,Cellphone only
3,9nphm,67,Bachelor's degree,Retired,White,Male,2,Owned or being bought by you or someone in you...,A one-family house attached to one or more houses,Somewhat Conservative,150-175K+,Internet Household,Iowa,Married,Metro Area,Don't Lean/Independent/None,Cellphone only
4,9nphm,79,Post grad study/professional degree,Retired,White,Male,2,Owned or being bought by you or someone in you...,A one-family house detached from any other house,Moderate,40-49K,Internet Household,Indiana,Married,Metro Area,Strong Democrat,Cellphone only


merging demographic and dataset together

In [14]:
demographicDataframe.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 17 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   study_id           100000 non-null  str  
 1   age                100000 non-null  int64
 2   education          100000 non-null  str  
 3   employment         100000 non-null  str  
 4   ethnicity          100000 non-null  str  
 5   gender             100000 non-null  str  
 6   household_size     100000 non-null  int64
 7   housing_ownership  100000 non-null  str  
 8   housing_type       100000 non-null  str  
 9   ideology           100000 non-null  str  
 10  income             100000 non-null  str  
 11  internet_access    100000 non-null  str  
 12  location           100000 non-null  str  
 13  marital_status     100000 non-null  str  
 14  metro_status       100000 non-null  str  
 15  party_id           100000 non-null  str  
 16  phone_service      100000 non-null  str  
dtypes: 

In [15]:
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 10 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   sample_id      100000 non-null  int64 
 1   participant    100000 non-null  int64 
 2   demographic    100000 non-null  object
 3   stimuli        100000 non-null  str   
 4   response       100000 non-null  int64 
 5   condition_num  100000 non-null  int64 
 6   task_num       100000 non-null  int64 
 7   prompt         100000 non-null  str   
 8   reasoning      100000 non-null  str   
 9   study_id       100000 non-null  str   
dtypes: int64(5), object(1), str(4)
memory usage: 244.7+ MB


In [16]:
dataset = pd.merge(dataset, 
         demographicDataframe.drop_duplicates(subset='study_id'), 
         on='study_id')
dataset.head()

,sample_id,participant,demographic,stimuli,response,condition_num,task_num,prompt,reasoning,study_id,...,housing_ownership,housing_type,ideology,income,internet_access,location,marital_status,metro_status,party_id,phone_service
0,0,0,"{'age': 34, 'education': 'Post grad study/prof...","You read ""Jaime is 20 years old and was born a...",5,7,0,You are a survey respondent with the following...,"Upon encountering the information about Jaime,...",9nphm,...,Owned or being bought by you or someone in you...,A building with 2 or more apartments,Moderate,75-99K,Internet Household,New Jersey,Married,Metro Area,Moderate Democrat,Cellphone only
1,1,0,"{'age': 34, 'education': 'Post grad study/prof...","You read ""Jaime is 20 years old and was born a...",3,7,1,You are a survey respondent with the following...,"Upon reading the scenario about Jaime, the res...",9nphm,...,Owned or being bought by you or someone in you...,A building with 2 or more apartments,Moderate,75-99K,Internet Household,New Jersey,Married,Metro Area,Moderate Democrat,Cellphone only
2,2,1,"{'age': 67, 'education': 'Bachelor's degree', ...","You read ""Jaime is 20 years old and was born a...",6,6,0,You are a survey respondent with the following...,"Upon reading the description of Jaime, this re...",9nphm,...,Owned or being bought by you or someone in you...,A building with 2 or more apartments,Moderate,75-99K,Internet Household,New Jersey,Married,Metro Area,Moderate Democrat,Cellphone only
3,3,1,"{'age': 67, 'education': 'Bachelor's degree', ...","You read ""Jaime is 20 years old and was born a...",5,6,1,You are a survey respondent with the following...,As an individual in their late 60s with a some...,9nphm,...,Owned or being bought by you or someone in you...,A building with 2 or more apartments,Moderate,75-99K,Internet Household,New Jersey,Married,Metro Area,Moderate Democrat,Cellphone only
4,4,2,"{'age': 79, 'education': 'Post grad study/prof...","You read ""Jaime is 20 years old and was born a...",6,4,0,You are a survey respondent with the following...,The respondent is likely to reflect on their o...,9nphm,...,Owned or being bought by you or someone in you...,A building with 2 or more apartments,Moderate,75-99K,Internet Household,New Jersey,Married,Metro Area,Moderate Democrat,Cellphone only


dropping useless columns

In [17]:
dataset = dataset[["stimuli","response","prompt","reasoning"]]

<h3 style="text-align:center"> Tokenisation/Fine tuning</h3>

In [18]:
dataset.head()

,stimuli,response,prompt,reasoning
0,"You read ""Jaime is 20 years old and was born a...",5,You are a survey respondent with the following...,"Upon encountering the information about Jaime,..."
1,"You read ""Jaime is 20 years old and was born a...",3,You are a survey respondent with the following...,"Upon reading the scenario about Jaime, the res..."
2,"You read ""Jaime is 20 years old and was born a...",6,You are a survey respondent with the following...,"Upon reading the description of Jaime, this re..."
3,"You read ""Jaime is 20 years old and was born a...",5,You are a survey respondent with the following...,As an individual in their late 60s with a some...
4,"You read ""Jaime is 20 years old and was born a...",6,You are a survey respondent with the following...,The respondent is likely to reflect on their o...


In [19]:
dataset["reasoning"].loc[0]

'Upon encountering the information about Jaime, a typical person in this persona may consider their own moderate stance towards gender identity issues and societal changes. They might reflect on the evolving nature of gender identity in contemporary society, recognizing that many individuals explore and change their identities over time. Given Jaime’s earlier feelings as a girl combined with the current non-binary identification, this respondent may weigh the likelihood of Jaime’s continued non-binary identification based on personal beliefs and societal acceptance. Additionally, the age factor plays a role in understanding that younger individuals may experience varied identity journeys. Overall, they may lean towards a belief that Jaime could still identify as non-binary in five years, but not without uncertainty.'

In [20]:
print(dataset["response"].min())
print(dataset["response"].max())

0
100


setting up model and tokenisation

In [21]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B", token=LLamaToken())
tokenizer.pad_token = tokenizer.eos_token #to be used when tokenizing the dataset

model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B", token=LLamaToken())

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

used to help identify the number of trainable paramaters in the llama model

In [22]:
def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    return f"trainable model parameters: {trainable_model_params}\nall model parameters: {all_model_params}\npercentage of trainable model parameters: {100 * trainable_model_params / all_model_params:.2f}%"

print(print_number_of_trainable_model_parameters(model))

trainable model parameters: 8030261248
all model parameters: 8030261248
percentage of trainable model parameters: 100.00%


splitting the dataset into 80% train and 20% test


In [23]:
testSize = 0.2
trainData, testData = train_test_split(dataset,random_state=42,test_size=testSize)
testDataAnalysis = testData.copy()
testData = testData.drop("response", axis=1)

del dataset

In [24]:
trainData.info()

<class 'pandas.DataFrame'>
Index: 80000 entries, 75220 to 15795
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   stimuli    80000 non-null  str  
 1   response   80000 non-null  int64
 2   prompt     80000 non-null  str  
 3   reasoning  80000 non-null  str  
dtypes: int64(1), str(3)
memory usage: 192.3 MB


In [25]:
testDataAnalysis.info()

<class 'pandas.DataFrame'>
Index: 20000 entries, 75721 to 42410
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   stimuli    20000 non-null  str  
 1   response   20000 non-null  int64
 2   prompt     20000 non-null  str  
 3   reasoning  20000 non-null  str  
dtypes: int64(1), str(3)
memory usage: 48.1 MB


This saves the dataframe to disk since converting it straight from pandas to hugging face causes memory issues (Unfortunatley i don't have 512GB RAM nor at the time of writing have the budget to buy that :) 

In [26]:
trainData.to_parquet("dataset/train_dataset.parquet")
testData.to_parquet("dataset/test_dataset.parquet")
testDataAnalysis.to_parquet("dataset/test_dataset_analysis.parquet")

In [27]:
del trainData, testData, testDataAnalysis

this handles tokenization

In [28]:
def preprocess_function(examples):
    combined = [
        f"Prompt: {p}\nStimuli: {s}\nReasoning: {r}"
        for p,s,r in zip(examples["prompt"], examples["stimuli"], examples["reasoning"])
    ]
    
    tokenized = tokenizer(
        combined,
        truncation=True,
        padding="max_length",
        max_length=512,          
        return_tensors=None      # Keep as lists for .map() compatibility
    )
    
    tokenized["labels"] = [ids.copy() for ids in tokenized["input_ids"]]
    
    return tokenized
    
# Tokenize the full dataset

trainData = Dataset.from_parquet("dataset/train_dataset.parquet")
tokenized_train_dataset = trainData.map(
    preprocess_function,
    num_proc=os.cpu_count() - 1,
    batched=True,   # your function already handles batches, so this works
    batch_size=1000
)

testData = Dataset.from_parquet("dataset/test_dataset.parquet")
tokenized_test_dataset = trainData.map(
    preprocess_function,
    num_proc=os.cpu_count() - 1,
    batched=True,   # your function already handles batches, so this works
    batch_size=1000
)

Generating train split: 0 examples [00:00, ? examples/s]

Map (num_proc=15):   0%|          | 0/80000 [00:00<?, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

<h2>carrying out fine tuning using SFT without LoRA or QLoRA</h2>

for LoRA

In [29]:

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

bitsandbytes library load error: Configured CUDA binary not found at /home/jambo/miniconda3/envs/UROP-project/lib/python3.12/site-packages/bitsandbytes/libbitsandbytes_cuda132.so
Traceback (most recent call last):
  File "/home/jambo/miniconda3/envs/UROP-project/lib/python3.12/site-packages/bitsandbytes/cextension.py", line 320, in <module>
    lib = get_native_library()
          ^^^^^^^^^^^^^^^^^^^^
  File "/home/jambo/miniconda3/envs/UROP-project/lib/python3.12/site-packages/bitsandbytes/cextension.py", line 288, in get_native_library
    raise RuntimeError(f"Configured {BNB_BACKEND} binary not found at {cuda_binary_path}")
RuntimeError: Configured CUDA binary not found at /home/jambo/miniconda3/envs/UROP-project/lib/python3.12/site-packages/bitsandbytes/libbitsandbytes_cuda132.so


trainable params: 3,407,872 || all params: 8,033,669,120 || trainable%: 0.0424


In [ ]:
sft_config = SFTConfig(
    output_dir="./llama-socsci-lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    max_length=2048,
    dataset_text_field="text",
    bf16=True,
    logging_steps=25,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    completion_only_loss=True           # still valid in 1.6
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset
)

results = trainer.train()
trainer.save_model("./llama-socsci-lora")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/tmp/ipykernel_7878/1160312305.py:1: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  sft_config = SFTConfig(


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,0.299994,0.287741,0.296807,0.910051,40960000.000000
2,0.285334,0.278895,0.285815,0.912267,81920000.000000


/home/jambo/miniconda3/envs/UROP-project/lib/python3.12/site-packages/peft/utils/other.py:1419: UserWarning: Unable to fetch remote file due to the following error 401 Client Error. (Request ID: Root=1-6a404e2c-116be1ff7df9711a2f8678e7;2cd6cd65-df5d-4bef-9a8b-9c195063d6ce)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.1-8B/resolve/main/config.json.
Access to model meta-llama/Llama-3.1-8B is restricted. You must have access to it and be authenticated to access it. Please log in. - silently ignoring the lookup for the file config.json in meta-llama/Llama-3.1-8B.
  warnings.warn(
/home/jambo/miniconda3/envs/UROP-project/lib/python3.12/site-packages/peft/utils/save_and_load.py:372: UserWarning: Could not find a config file in meta-llama/Llama-3.1-8B - will assume that the vocabulary was not modified.
  warnings.warn(
/home/jambo/miniconda3/envs/UROP-project/lib/python3.12/site-packages/peft/utils/other.py:1419: UserWarning: Unable to fetch remote file due to t

In [ ]:
import inspect
from trl import SFTConfig
print(inspect.signature(SFTConfig))

In [36]:
!pip install -U bitsandbytes

  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 2.5 MB/s  0:00:24m0:00:0100:01
